# Classic vs Serverless DBU Cost Comparison

Compares the actual billed DBU for a Databricks job run on **classic Job Compute** vs **Serverless Compute**, using `system.billing.usage` as the source of truth.

### Prerequisites
1. Run the job on **classic compute** for at least `RUNS` executions
2. Switch the job to **serverless compute** and let it run for at least `RUNS` executions
3. Fill in the four variables in the next cell and run all cells

In [ ]:
# ── Fill these in ─────────────────────────────────────────────
PROFILE      = "e2-demo-fe"        # Databricks CLI profile (~/.databrickscfg)
JOB_ID       = 262685974171222     # Job ID to evaluate
WAREHOUSE_ID = "e9b34f7a2e4b0561"  # Any running SQL warehouse in the workspace
RUNS         = 144                 # Number of runs to compare per compute type
# ──────────────────────────────────────────────────────────────

In [ ]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

client = WorkspaceClient(profile=PROFILE)

def run_sql(statement: str) -> list[dict]:
    resp = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID,
        statement=statement,
    )
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        time.sleep(1)
        resp = client.statement_execution.get_statement(resp.statement_id)
    if resp.status.state != StatementState.SUCCEEDED:
        raise RuntimeError(f"Query failed: {resp.status.error}")
    cols = [c.name for c in resp.manifest.schema.columns]
    return [dict(zip(cols, row)) for row in (resp.result.data_array or [])]

## Step 1 — Identify comparison windows

Pulls the last `RUNS` classic runs and the first `RUNS` serverless runs (starting from when classic ended, to exclude any pre-deployment test runs) directly from the billing table.

In [ ]:
run_id_sql = f"""
WITH classic_cutoff AS (
  SELECT MAX(usage_start_time) AS t
  FROM system.billing.usage
  WHERE usage_metadata.job_id = '{JOB_ID}'
    AND product_features.is_serverless = false
    AND record_type = 'ORIGINAL'
),
classic_runs AS (
  SELECT usage_metadata.job_run_id AS run_id, usage_start_time,
         ROW_NUMBER() OVER (ORDER BY usage_start_time DESC) AS rn
  FROM system.billing.usage
  WHERE usage_metadata.job_id = '{JOB_ID}'
    AND product_features.is_serverless = false
    AND record_type = 'ORIGINAL'
),
serverless_runs AS (
  SELECT u.usage_metadata.job_run_id AS run_id, u.usage_start_time,
         ROW_NUMBER() OVER (ORDER BY u.usage_start_time ASC) AS rn
  FROM system.billing.usage u
  CROSS JOIN classic_cutoff c
  WHERE u.usage_metadata.job_id = '{JOB_ID}'
    AND u.product_features.is_serverless = true
    AND u.record_type = 'ORIGINAL'
    AND u.usage_start_time >= c.t
)
SELECT 'classic'    AS compute_type, run_id, usage_start_time FROM classic_runs    WHERE rn <= {RUNS}
UNION ALL
SELECT 'serverless' AS compute_type, run_id, usage_start_time FROM serverless_runs WHERE rn <= {RUNS}
"""

run_rows = run_sql(run_id_sql)

classic_ids    = [r["run_id"] for r in run_rows if r["compute_type"] == "classic"]
serverless_ids = [r["run_id"] for r in run_rows if r["compute_type"] == "serverless"]

classic_times    = sorted(r["usage_start_time"] for r in run_rows if r["compute_type"] == "classic")
serverless_times = sorted(r["usage_start_time"] for r in run_rows if r["compute_type"] == "serverless")

print(f"Classic    — {len(classic_ids):>3} runs  |  {classic_times[0][:16]} → {classic_times[-1][:16]}")
print(f"Serverless — {len(serverless_ids):>3} runs  |  {serverless_times[0][:16]} → {serverless_times[-1][:16]}")

if len(classic_ids) < RUNS:
    print(f"\n⚠️  Only {len(classic_ids)} classic runs found — need at least {RUNS}. Run the job on classic compute longer.")
if len(serverless_ids) < RUNS:
    print(f"\n⚠️  Only {len(serverless_ids)} serverless runs found — need at least {RUNS}. Run the job on serverless longer.")

## Step 2 — Compare billed DBU

In [ ]:
all_ids  = classic_ids + serverless_ids
id_list  = ", ".join(f"'{rid}'" for rid in all_ids)

billing_sql = f"""
SELECT
  CASE WHEN product_features.is_serverless THEN 'Serverless' ELSE 'Classic' END AS compute_type,
  sku_name,
  COUNT(*)                       AS runs,
  ROUND(SUM(usage_quantity), 6)  AS total_dbu,
  MIN(usage_start_time)          AS window_start,
  MAX(usage_start_time)          AS window_end
FROM system.billing.usage
WHERE usage_metadata.job_run_id IN ({id_list})
  AND record_type = 'ORIGINAL'
GROUP BY 1, 2
ORDER BY 1
"""

rows   = run_sql(billing_sql)
by_type = {r["compute_type"]: r for r in rows}
c, s   = by_type.get("Classic", {}), by_type.get("Serverless", {})

c_dbu  = float(c.get("total_dbu", 0))
s_dbu  = float(s.get("total_dbu", 0))
ratio  = c_dbu / s_dbu if s_dbu else float("inf")

print()
print("=" * 66)
print(f"  DBU COMPARISON  —  job {JOB_ID}  ({RUNS} runs each)")
print("=" * 66)
print(f"  {'':30s}  {'Classic':>10}  {'Serverless':>10}")
print(f"  {'-'*30}  {'-'*10}  {'-'*10}")
print(f"  {'Runs':30s}  {c.get('runs', '—'):>10}  {s.get('runs', '—'):>10}")
print(f"  {'SKU':30s}  {'JOBS_COMPUTE':>10}  {'SERVERLESS':>10}")
print(f"  {'Window start':30s}  {str(c.get('window_start',''))[:10]:>10}  {str(s.get('window_start',''))[:10]:>10}")
print(f"  {'Window end':30s}  {str(c.get('window_end',''))[:10]:>10}  {str(s.get('window_end',''))[:10]:>10}")
print(f"  {'-'*30}  {'-'*10}  {'-'*10}")
print(f"  {'Total billed DBU':30s}  {c_dbu:>10.4f}  {s_dbu:>10.4f}")
print(f"  {'Serverless savings':30s}  {ratio:>10.2f}x cheaper")
print("=" * 66)
print()